### This notebook is a test of "of_bpdrr.py" as a module that contains the functions of all 6 objective functions of the Battle of Postdisaster Response and Restoration

In [18]:
# Import libraries to be used, including "of_bpdrr", which contains the objective fuctions of the BPDRR
import of_bpdrr
import wntr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [19]:
# Import the water network model
ds_sel = 'DS2' # which damage scenario to analyse
#inp_file = "BBM-EPS_"+ds_sel+"mcg.inp"
inp_file = "BBM-EPS_"+ds_sel+"_restoration.inp"
wn = wntr.network.WaterNetworkModel(inp_file)

In [20]:
# Simulation settings
wn.options.time.duration = 7 * 24 * 3600       # 7 days
wn.options.time.hydraulic_timestep = 15*60     # 15 minutes
wn.options.time.report_timestep = 15*60        # 15 minutes

# Start simulation at 07:00 AM
wn.options.time.start_clocktime = 7 * 3600

# Pressure Driven Analysis
wn.options.hydraulic.demand_model = "PDA"
wn.options.hydraulic.required_pressure = 20
wn.options.hydraulic.minimum_pressure = 0
wn.options.hydraulic.pressure_exponent = 0.5

In [21]:
# timestep
dt = wn.options.time.hydraulic_timestep

# timestep in minutes
dtm = dt/60

In [22]:
# run the simulation
sim = wntr.sim.EpanetSimulator(wn)
results = sim.run_sim()

In [23]:
# getting the actual demand at each node over time 
demand = results.node["demand"]*1000  # convert to L/s

In [24]:
# Find all demand nodes (base demand > 0)
demand_nodes = []

# Expected demand at each node
#expected = {}

for node_name in wn.junction_name_list:

    node = wn.get_node(node_name)

    if node.base_demand > 0:

        demand_nodes.append(node_name)

print(f"Demand nodes found: {len(demand_nodes)}")

Demand nodes found: 4201


In [25]:
# Expected demand
expected = wntr.metrics.expected_demand(wn)

expected *= 1000        # convert to L/s

expected = expected[demand_nodes]

expected.index = demand.index

In [26]:
# Calculate the supply ratio at each time step
supply_ratio = demand[demand_nodes] / expected

# Clip the supply ratio to be between 0 and 1
supply_ratio = supply_ratio.clip(lower=0, upper=1)

In [27]:
# Finding emitter nodes (nodes starting with "E_")
emitter_nodes = [
    j for j in wn.junction_name_list
    if j.startswith("E_")
]

# Demand at emitter nodes (leakage) over time
leakage = demand[emitter_nodes]

### Functionality 
The percentage of the demand supplied by the distribution system.

In [28]:
functionality_series = of_bpdrr.system_functionality(demand, demand_nodes, expected)

print(functionality_series)

0         100.0
900       100.0
1800      100.0
2700      100.0
3600      100.0
          ...  
601200    100.0
602100    100.0
603000    100.0
603900    100.0
604800    100.0
Length: 673, dtype: float64


### 1. OF: Time without supply for hospital/firefighting (FH)
The time that the hospitals and the firefighting flows are without supply It's calculated by multiplying the simulation time step duration  with the number of time steps in which the supply/demand ratio for the hospitals and firefighting flows was less than 0.5.

In [29]:
FH_value, FH_undersupplied = of_bpdrr.OF_FH(supply_ratio, dtm)

print(f"FH = {FH_value:,.2f} minutes") 

FH = 660.00 minutes


### 2. rapidity of recovery (t95)
The time that the hospitals and the firefighting flows are without supply It's calculated by multiplying the simulation time step duration  with the number of time steps in which the supply/demand ratio for the hospitals and firefighting flows was less than 0.5.

In [30]:
t95 = of_bpdrr.OF_t95(demand, demand_nodes, expected, dtm)

print(f"Rapidity of recovery (t95): {t95:.2f}")

Rapidity of recovery (t95): 5460.00


### 3. OF: Resilience loss (RL)
RL considers the Accumulated loss of functionality during the recovery process of the system. It represents the area between the full functionality line (100%) and the functionality time series

In [31]:
RL = of_bpdrr.OF_RL(demand, demand_nodes, expected, dtm)

print(f"Resilience Loss = {RL:,.2f} %-minutes")

Resilience Loss = 19,615.90 %-minutes


### 4. OF: Average time of no user service (Time no serv.)
Measures the average time each demand node across the network stayed without service. The formula of this objective is similar to the FH formula, but it considers the sum of all demand nodes that were undersupplied (supply/demand ratio under 0.5) divided by the total amount of demand nodes (DN)

In [32]:
time_no_serv = of_bpdrr.OF_time_no_serv(supply_ratio, dtm)

print(f"Average Time of No User Service = {time_no_serv:.2f} minutes")

Average Time of No User Service = 46.63 minutes


### 5. OF: Number of users without service for eight consecutive hours (NWSECH)
Number of demand nodes that stayed without service for more than eight consecutive hours. It’s calculated by counting the amount of nodes with at least one continuous 8-hour period during which the node's supply/demand ratio never exceeds 0.5.

In [33]:
NNS = of_bpdrr.OF_NWSECH(supply_ratio, dtm)

print(f"Nodes without service for 8 consecutive hours = {NNS}")

Nodes without service for 8 consecutive hours = 2


### 6. OF: Water loss (WL)
counts the volume in litres of water lost during the period after the earthquake, by multiplying the time step with the sum of the outflows across all damages in the system.

In [34]:
WL = of_bpdrr.OF_WL(demand, emitter_nodes, dt)

print(f"Water loss = {WL:,.2f} m³")

Water loss = 76,657.73 m³
